In [1]:
import os, sys, platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())

# core libs
import geopandas as gpd
import fiona
import shapely
import pyproj

print("geopandas:", gpd.__version__)
print("fiona:", fiona.__version__)
print("shapely:", shapely.__version__)
print("pyproj:", pyproj.__version__)

# confirm we see the project filesystem
print("\n/work listing (top):")
print(sorted(os.listdir("/work"))[:20])

print("\n/work/data listing:")
print(sorted(os.listdir("/work/data"))[:50])

print("\nnormalized folders:")
print(sorted(os.listdir("/work/data/normalized")))


Python: 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Platform: Linux-6.8.0-100-generic-x86_64-with-glibc2.35
CWD: /work
geopandas: 1.1.2
fiona: 1.10.1
shapely: 2.1.2
pyproj: 3.7.2

/work listing (top):
['.env.tsird', '.gitignore', '.ipynb_checkpoints', 'COMPLETION.md', 'DEPLOYMENT_CHECKLIST.txt', 'PROJECT_COMPLETION_REPORT.md', 'README.md', 'TSIRD-Data-Cleaning.ipynb', 'backups', 'data', 'db', 'docker-compose.yml', 'docs', 'etl', 'infra', 'scripts', 'tmp', 'ui']

/work/data listing:
['gold', 'logs', 'normalized', 'processed', 'raw', 'source', 'staging']

normalized folders:
['normalize_4326.log', 'vectors4326', 'vectors4326_final', 'vectors4326_fixed']


In [2]:
import os, glob, math
import geopandas as gpd

BASE = "/work/data/normalized"
CANDIDATE_DIRS = ["vectors4326_final", "vectors4326_fixed", "vectors4326"]

def scan_dir(d):
    path = os.path.join(BASE, d)
    if not os.path.isdir(path):
        return []
    shp_files = sorted(glob.glob(os.path.join(path, "*.shp")))
    results = []
    for shp in shp_files:
        name = os.path.basename(shp)
        try:
            gdf = gpd.read_file(shp)
            crs = str(gdf.crs) if gdf.crs else "UNKNOWN"
            minx, miny, maxx, maxy = gdf.total_bounds

            # basic numeric sanity
            finite = all(map(math.isfinite, [minx, miny, maxx, maxy]))

            # Ethiopia-ish bounds sanity in EPSG:4326 terms
            # lon approx [20..55], lat approx [-5..20]
            looks_4326 = (
                finite and
                -180 <= minx <= 180 and -90 <= miny <= 90 and
                -180 <= maxx <= 180 and -90 <= maxy <= 90
            )
            looks_ethiopia = looks_4326 and (20 <= minx <= 55) and (-5 <= miny <= 20) and (20 <= maxx <= 55) and (-5 <= maxy <= 20)

            results.append({
                "dir": d,
                "file": name,
                "crs": crs,
                "features": len(gdf),
                "bounds": (round(minx,6), round(miny,6), round(maxx,6), round(maxy,6)),
                "finite_bounds": finite,
                "looks_like_deg": looks_4326,
                "looks_like_ethiopia": looks_ethiopia,
            })
        except Exception as e:
            results.append({
                "dir": d,
                "file": name,
                "crs": "ERROR",
                "features": None,
                "bounds": None,
                "finite_bounds": False,
                "looks_like_deg": False,
                "looks_like_ethiopia": False,
                "error": repr(e),
            })
    return results

rows = []
for d in CANDIDATE_DIRS:
    rows.extend(scan_dir(d))

# print a compact report: anything suspicious bubbles to the top
def is_suspicious(r):
    if r.get("crs") in ("UNKNOWN", "ERROR"):
        return True
    if not r.get("finite_bounds"):
        return True
    # if it claims EPSG:4326 but doesn't look like degrees, that's suspicious
    if "4326" in r.get("crs","") and not r.get("looks_like_deg"):
        return True
    # if it doesn't look like Ethiopia at all, suspicious for an Ethiopia/Tigray atlas
    if "4326" in r.get("crs","") and not r.get("looks_like_ethiopia"):
        return True
    return False

sus = [r for r in rows if is_suspicious(r)]
ok  = [r for r in rows if not is_suspicious(r)]

print(f"Total shapefiles scanned: {len(rows)}")
print(f"OK: {len(ok)}")
print(f"Suspicious: {len(sus)}\n")

print("=== Suspicious (review these first) ===")
for r in sus[:80]:
    print(f"[{r['dir']}] {r['file']}: crs={r['crs']} features={r['features']} bounds={r['bounds']}" + (f" error={r.get('error')}" if r.get("error") else ""))

print("\n=== Sample OK ===")
for r in ok[:20]:
    print(f"[{r['dir']}] {r['file']}: crs={r['crs']} features={r['features']} bounds={r['bounds']}")


Total shapefiles scanned: 40
OK: 35
Suspicious: 5

=== Suspicious (review these first) ===
[vectors4326_fixed] TigrayHealth2006_4326_fixed.shp: crs=EPSG:4326 features=729 bounds=(-141.681925, 1.251743, 43.617006, 14.729095)
[vectors4326] TigrayHealth2006_4326.shp: crs=UNKNOWN features=729 bounds=(226852.0, 138155.0, 999868.0, 5.832460226579078e+238)
[vectors4326] TigrayRoadsIn2006_4326.shp: crs=UNKNOWN features=97 bounds=(216984.791281, 1357397.999842, 594406.833049, 1628341.000158)
[vectors4326] ethio_wereda_4326.shp: crs=EPSG:4326 features=466 bounds=(-161882.5625, 376388.125, 1495282.875, 1645935.625)
[vectors4326] ethio_wereda_Project_4326.shp: crs=ERROR features=None bounds=None error=UnicodeDecodeError('utf-8', b'\xaa\xf5*A', 0, 1, 'invalid start byte')

=== Sample OK ===
[vectors4326_fixed] TigrayHealth2006_xy_4326_fixed.shp: crs=EPSG:4326 features=729 bounds=(36.470582, 0.001845, 39.924149, 14.729095)
[vectors4326_fixed] TigrayRoadsIn2006_4326_fixed.shp: crs=EPSG:4326 features=

In [4]:
mkdir -p "/work/Docker Projects/TSIRD-Atlas-Data-Pipeline/reports/audit"

mkdir: cannot create directory ‘/work/Docker Projects/TSIRD-Atlas-Data-Pipeline/reports/audit’: Permission denied
